In [3]:
import pandas as pd

# Cargamos la metadata de un componente (ej. Nova)
# Nota: Si es .tsv, el separador es un tabulador '\t'
df_meta = pd.read_csv('Fault-Injection-Dataset-master\\nova.csv', sep='\t')

print("Dimensiones de la metadata:", df_meta.shape)
print(df_meta.head())

Dimensiones de la metadata: (439, 1)
  Test,Round_1 Failure,Round_2 Failure
0                        Test_1,yes,no
1                         Test_2,no,no
2                       Test_3,yes,yes
3                        Test_4,yes,no
4                        Test_5,yes,no


In [5]:
with open('Fault-Injection-Dataset-master\\Nova\\Test_1\\logs\\round_1\\nova\\nova-compute.log.bzip2.out', 'r') as f:
    for i in range(5):
        print(f.readline())

2018-06-26 03:21:12.936 1867 DEBUG nova.compute.resource_tracker [req-75a12c6d-df58-46c2-b64a-5b4bc1516aea - - - - -] Total usable vcpus: 8, total allocated vcpus: 0 _report_final_resource_view /usr/lib/python2.7/site-packages/nova/compute/resource_tracker.py:839

2018-06-26 03:21:12.937 1867 INFO nova.compute.resource_tracker [req-75a12c6d-df58-46c2-b64a-5b4bc1516aea - - - - -] Final resource view: name=localhost.localdomain phys_ram=16383MB used_ram=512MB phys_disk=143GB used_disk=0GB total_vcpus=8 used_vcpus=0 pci_stats=[]

2018-06-26 03:21:13.016 1867 DEBUG nova.scheduler.client.report [req-75a12c6d-df58-46c2-b64a-5b4bc1516aea - - - - -] Refreshing aggregate associations for resource provider df00d0eb-582c-42fa-b99e-e6c7965395d9 _ensure_resource_provider /usr/lib/python2.7/site-packages/nova/scheduler/client/report.py:509

2018-06-26 03:21:13.082 1867 DEBUG nova.compute.resource_tracker [req-75a12c6d-df58-46c2-b64a-5b4bc1516aea - - - - -] Compute_service record updated for localhos

In [ ]:
import re

def parsear_linea_log(linea):
    # RegEx diseñada específicamente para el formato de OpenStack
    patron = r'^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{3})\s+(\d+)\s+(\w+)\s+([\w\.]+)\s+(\[.*?\])\s+(.+)$'
    
    match = re.match(patron, linea.strip())
    if match:
        return {
            'timestamp': match.group(1),
            'pid': int(match.group(2)),
            'level': match.group(3),
            'module': match.group(4),
            'request_id': match.group(5).strip('[]'), # Quitamos los corchetes para limpiar el ID
            'message': match.group(6)
        }
    return None

linea_ejemplo = "2018-06-26 03:21:12.936 1867 DEBUG nova.compute.resource_tracker [req-75a12c6d-df58-46c2-b64a-5b4bc1516aea - - - - -] Total usable vcpus: 8, total allocated vcpus: 0 _report_final_resource_view /usr/lib/python2.7/site-packages/nova/compute/resource_tracker.py:839"

resultado = parsear_linea_log(linea_ejemplo)
import json
print(json.dumps(resultado, indent=4))

{
    "timestamp": "2018-06-26 03:21:12.936",
    "pid": 1867,
    "level": "DEBUG",
    "module": "nova.compute.resource_tracker",
    "request_id": "req-75a12c6d-df58-46c2-b64a-5b4bc1516aea - - - - -",
    "message": "Total usable vcpus: 8, total allocated vcpus: 0 _report_final_resource_view /usr/lib/python2.7/site-packages/nova/compute/resource_tracker.py:839"
}


In [11]:
def log_to_dataframe(ruta_archivo):
    lineas_parseadas = []
    
    with open(ruta_archivo, 'r') as f:
        for linea in f:
            datos_linea = parsear_linea_log(linea)
            if datos_linea: # Si la línea coincidió con el patrón
                lineas_parseadas.append(datos_linea)
                
    # Convertimos la lista de diccionarios en un DataFrame listo para analizar
    return pd.DataFrame(lineas_parseadas)

df_logs = log_to_dataframe('Fault-Injection-Dataset-master\\Nova\\Test_1\\logs\\round_1\\nova\\nova-compute.log.bzip2.out')
print(df_logs.head())

                 timestamp   pid  level                         module  \
0  2018-06-26 03:21:12.936  1867  DEBUG  nova.compute.resource_tracker   
1  2018-06-26 03:21:12.937  1867   INFO  nova.compute.resource_tracker   
2  2018-06-26 03:21:13.016  1867  DEBUG   nova.scheduler.client.report   
3  2018-06-26 03:21:13.082  1867  DEBUG  nova.compute.resource_tracker   
4  2018-06-26 03:21:13.084  1867  DEBUG     oslo_concurrency.lockutils   

                                          request_id  \
0  req-75a12c6d-df58-46c2-b64a-5b4bc1516aea - - -...   
1  req-75a12c6d-df58-46c2-b64a-5b4bc1516aea - - -...   
2  req-75a12c6d-df58-46c2-b64a-5b4bc1516aea - - -...   
3  req-75a12c6d-df58-46c2-b64a-5b4bc1516aea - - -...   
4  req-75a12c6d-df58-46c2-b64a-5b4bc1516aea - - -...   

                                             message  
0  Total usable vcpus: 8, total allocated vcpus: ...  
1  Final resource view: name=localhost.localdomai...  
2  Refreshing aggregate associations for resource... 

In [16]:
import re
import pandas as pd
from pathlib import Path

def parsear_linea_log(linea):
    # Mantenemos nuestra RegEx robusta para el formato Oslo de OpenStack
    patron = r'^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{3})\s+(\d+)\s+(\w+)\s+([\w\.]+)\s+(\[.*?\])\s+(.+)$'
    match = re.match(patron, linea.strip())
    if match:
        return {
            'timestamp': match.group(1),
            'pid': int(match.group(2)),
            'level': match.group(3),
            'module': match.group(4),
            'request_id': match.group(5).strip('[]'),
            'message': match.group(6)
        }
    return None

def cargar_muestra_del_dataset(ruta_raiz, limite_tests=100):
    ruta_base = Path(ruta_raiz)
    todos_los_logs = []
    componentes = ['Nova', 'Cinder', 'Neutron']
    
    for comp in componentes:
        ruta_comp = ruta_base / comp
        if not ruta_comp.exists():
            continue
            
        print(f"Procesando componente: {comp}...")
        
        carpetas_test = list(ruta_comp.glob('Test_*'))[:limite_tests]
        print(f"   -> Encontrados {len(carpetas_test)} tests para procesar (Límite aplicado).")
        
        for carpeta_test in carpetas_test:
            test_id = carpeta_test.name  
            
            for archivo_log in carpeta_test.rglob('*log*'):
                if archivo_log.is_dir():
                    continue
                
                partes_ruta = archivo_log.parts
                round_id = next((p for p in partes_ruta if 'round_' in p), 'no_round')
                
                lineas_parseadas = []
                with open(archivo_log, 'r', encoding='utf-8', errors='ignore') as f:
                    for linea in f:
                        datos = parsear_linea_log(linea)
                        if datos:
                            datos['main_component'] = comp
                            datos['test_id'] = test_id
                            datos['round_id'] = round_id
                            datos['log_file_name'] = archivo_log.name
                            lineas_parseadas.append(datos)
                
                if lineas_parseadas:
                    df_temporal = pd.DataFrame(lineas_parseadas)
                    todos_los_logs.append(df_temporal)
                    
    if todos_los_logs:
        print("Unificando la muestra de logs")
        df_final = pd.concat(todos_los_logs, ignore_index=True)
        print(f"Muestra lista con {len(df_final)} líneas de log estructuradas.")
        return df_final
    else:
        return pd.DataFrame()


ruta_proyecto = 'Fault-Injection-Dataset-master'
# Extraemos solo 50 tests por componente para empezar ligeros
df_logs_muestra = cargar_muestra_del_dataset(ruta_proyecto, limite_tests=30)

# Si todo sale bien, inspecciona las primeras filas
if not df_logs_muestra.empty:
    print(df_logs_muestra.head())

Procesando componente: Nova...
   -> Encontrados 30 tests para procesar (Límite aplicado).
Procesando componente: Cinder...
   -> Encontrados 30 tests para procesar (Límite aplicado).
Procesando componente: Neutron...
   -> Encontrados 30 tests para procesar (Límite aplicado).
Unificando la muestra de logs
Muestra lista con 1430326 líneas de log estructuradas.
                 timestamp   pid    level                         module  \
0  2018-06-26 03:27:39.573  2283    DEBUG           eventlet.wsgi.server   
1  2018-06-26 03:27:39.606  2283  WARNING  keystonemiddleware.auth_token   
2  2018-06-26 03:27:40.845  2283    DEBUG     oslo_policy._cache_handler   
3  2018-06-26 03:27:40.859  2283    DEBUG             oslo_policy.policy   
4  2018-06-26 03:27:41.558  2283     INFO      cinder.api.openstack.wsgi   

                                          request_id  \
0                                                  -   
1                                                  -   
2  req-eeb6f